# 06 — Risk Simulation

This notebook inspects normalized public-sample PnL using bootstrap-style Monte Carlo simulation.

The goal is not to build a live capital allocation model. The goal is to show how sparse public-sample outcomes can produce path-dependent risk once execution frictions are included.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "sample"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

pd.set_option("display.max_columns", 80)


In [ ]:
from risk.monte_carlo import load_normalized_pnl, summarize_monte_carlo, bootstrap_paths

pnl_values = load_normalized_pnl(DATA_DIR / "settlements_sample.csv")
summary = summarize_monte_carlo(pnl_values, simulation_count=1000, seed=42)

print(f"normalized PnL observations: {len(pnl_values):,}")
print(f"mean final PnL: {summary.mean_final_pnl:.4f}")
print(f"p05 final PnL: {summary.p05_final_pnl:.4f}")
print(f"p95 final PnL: {summary.p95_final_pnl:.4f}")


## Input PnL distribution

The public sample uses normalized market-level PnL. It is not real account PnL and does not reconstruct full private execution state.


In [ ]:
pnl = pd.Series(pnl_values, name="net_pnl_normalized")
pd.DataFrame(
    [
        {"metric": "count", "value": pnl.count()},
        {"metric": "mean", "value": pnl.mean()},
        {"metric": "median", "value": pnl.median()},
        {"metric": "positive_rate", "value": (pnl > 0).mean()},
        {"metric": "zero_rate", "value": (pnl == 0).mean()},
        {"metric": "min", "value": pnl.min()},
        {"metric": "max", "value": pnl.max()},
    ]
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(pnl, bins=40)
ax.set_title("Normalized public-sample PnL distribution")
ax.set_xlabel("net_pnl_normalized")
ax.set_ylabel("Markets")
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## Monte Carlo summary

The simulation resamples normalized public-sample PnL values with replacement. This preserves the sample outcome distribution while randomizing path order.


In [ ]:
mc_summary = pd.DataFrame(
    [
        {"metric": "simulation_count", "value": summary.simulation_count},
        {"metric": "path_horizon", "value": summary.horizon},
        {"metric": "mean_final_pnl", "value": summary.mean_final_pnl},
        {"metric": "median_final_pnl", "value": summary.median_final_pnl},
        {"metric": "p05_final_pnl", "value": summary.p05_final_pnl},
        {"metric": "p95_final_pnl", "value": summary.p95_final_pnl},
        {"metric": "mean_max_drawdown", "value": summary.mean_max_drawdown},
        {"metric": "p95_max_drawdown", "value": summary.p95_max_drawdown},
        {"metric": "mean_longest_losing_streak", "value": summary.mean_longest_losing_streak},
        {"metric": "p95_longest_losing_streak", "value": summary.p95_longest_losing_streak},
    ]
)
mc_summary


In [ ]:
paths = bootstrap_paths(pnl_values, simulation_count=1000, seed=42)
terminal = pd.Series([path.final_pnl for path in paths], name="terminal_pnl")
drawdown = pd.Series([path.max_drawdown for path in paths], name="max_drawdown")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(terminal, bins=40)
ax.set_title("Monte Carlo terminal normalized PnL")
ax.set_xlabel("terminal PnL")
ax.set_ylabel("Simulated paths")
ax.grid(True, axis="y", alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(drawdown, bins=40)
ax.set_title("Monte Carlo max drawdown")
ax.set_xlabel("max drawdown")
ax.set_ylabel("Simulated paths")
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## Author takeaway

Pure tick replay and simulated backtests can show positive edge, but the live-like ledger sample gives a different answer after failed order submission, latency, quote staleness, API variability, fill probability, and execution gates are included. The seven-day public sample is short and live-like, so it looks sparse and zero-inflated.

This simulation-to-live gap is the core experiment of the project: aligning replayed edge with realized executable performance.
